In [ ]:
from dataclasses import dataclass
import pickle
import gzip
import random
import numpy as np
from PIL import Image, ImageOps
from time import time

In [ ]:
import requests
import os

# URL for the MNIST dataset file
url = 'http://deeplearning.net/data/mnist/mnist.pkl.gz'
filename = 'mnist.pkl.gz'

# Check if the file already exists to avoid re-downloading
if not os.path.exists(filename):
    print(f'Downloading {filename}...')
    response = requests.get(url, stream=True)
    response.raise_for_status() # Raise an exception for HTTP errors

    with open(filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f'Downloaded {filename} successfully.')
else:
    print(f'{filename} already exists. Skipping download.')

mnist.pkl.gz already exists. Skipping download.


In [ ]:
def sigmoid(z):
  return 1.0/(1.0+np.exp(-z))

In [ ]:
def sigmoid_prime(z):
  return sigmoid(z)*(1-sigmoid(z))

In [ ]:
def cost_derivative(output_activations, y):
  return (output_activations - y)

In [ ]:

class Network:
    def __init__(self, num_layers, biases, weights):
        self.num_layers = num_layers
        self.biases = biases
        self.weights = weights

def init_network(layers):
  return Network(
      len(layers),
      [np.random.randn(y, 1) for y in layers[1:]],
      [np.random.randn(y, x) for x, y in zip(layers[:-1], layers[1:])]
  )

In [ ]:
def feedforward(nn, a):
  for b, w in zip(nn.biases, nn.weights):
    a = sigmoid(np.dot(w,a)+b)
  return a

In [ ]:
def evaluate(nn, test_data):
  test_results = [(np.argmax(feedforward(nn, x)),y) for (x, y) in test_data]
  return sum(int(x == y) for (x, y) in test_results)

In [ ]:
def learn(nn, training_data, epochs, mini_batch_size, learning_rate, test_data = None):
    n = len(training_data)

    for j in range(epochs):
        random.shuffle(training_data) # that's where "stochastic" comes from

        mini_batches = [
            training_data[k: k + mini_batch_size] for k in range(0, n, mini_batch_size)
        ]

        for mini_batch in mini_batches:
            stochastic_gradient_descent(nn, mini_batch, learning_rate) # that's where learning really happes

        if test_data:
            print('Epoch {0}: accuracy {1}%'.format(f'{j + 1:2}', 100.0 * evaluate(nn, test_data) / len(test_data)))
        else:
            print('Epoch {0} complete.'.format(f'{j + 1:2}'))

In [34]:
def stochastic_gradient_descent(nn, mini_batch, eta):
    """
    Updates the network's weights and biases by applying gradient descent
    using a fully matrix-based approach.
    """
    # Stack individual column vectors horizontally into 2D matrices
    # X shape becomes (784, mini_batch_size)
    # Y shape becomes (10, mini_batch_size)
    X = np.hstack([x for x, y in mini_batch])
    Y = np.hstack([y for x, y in mini_batch])

    # Compute the gradients for the entire batch simultaneously
    nabla_b, nabla_w = backprop(nn, X, Y)

    # Update weights and biases
    m = len(mini_batch)
    nn.weights = [w - (eta / m) * nw for w, nw in zip(nn.weights, nabla_w)]
    nn.biases  = [b - (eta / m) * nb for b, nb in zip(nn.biases, nabla_b)]

In [35]:
def backprop(nn, x, y):
    nabla_b = [np.zeros(b.shape) for b in nn.biases]
    nabla_w = [np.zeros(w.shape) for w in nn.weights]

    # feedforward
    activation = x    # first layer activation is just its input
    activations = [x] # list to store all activations, layer by layer
    zs = []           # list to store all z vectors, layer by layer

    for b, w in zip(nn.biases, nn.weights):
        z = np.dot(w, activation) + b  # calculate z for the current layer
        zs.append(z)                   # store
        activation = sigmoid(z)        # layer output
        activations.append(activation) # store

    # backward pass

    # 1. starting from the output layer
    delta = cost_derivative(activations[-1], y) * sigmoid_prime(zs[-1])
    nabla_b[-1] = np.sum(delta, axis=1, keepdims=True)
    nabla_w[-1] = np.dot(delta, activations[-2].transpose())

    # 2. continue back to the input layer (i is the layer index, we're using i instead of l
    #    to improve readability -- l looks too much like 1)
    for i in range(2, nn.num_layers): # starting from the next-to-last layer
        z = zs[-i]
        sp = sigmoid_prime(z)
        delta = np.dot(nn.weights[-i + 1].transpose(), delta) * sp

        nabla_b[-i] = np.sum(delta, axis=1, keepdims=True)
        nabla_w[-i] = np.dot(delta, activations[-i - 1].transpose())

    return (nabla_b, nabla_w)

In [36]:
def load_data():
  f=gzip.open('mnist.pkl.gz', 'rb')
  training_data, validation_data, test_data = pickle.load(f, encoding="bytes")
  f.close()

  return (training_data, validation_data, test_data)

In [37]:
def load_data_wrapper():
    tr_d, va_d, te_d = load_data()

    training_inputs = [np.reshape(x, (784, 1)) for x in tr_d[0]]
    training_results = [one_hot_encode(y) for y in tr_d[1]]
    training_data = zip(training_inputs, training_results)
    validation_inputs = [np.reshape(x, (784, 1)) for x in va_d[0]]
    validation_data = zip(validation_inputs, va_d[1])
    test_inputs = [np.reshape(x, (784, 1)) for x in te_d[0]]
    test_data = zip(test_inputs, te_d[1])

    return (list(training_data), list(validation_data), list(test_data))

In [38]:
def one_hot_encode(j):
  e = np.zeros((10, 1))
  e[j] = 1.0
  return e

In [39]:
training_data, validation_data, test_data = load_data_wrapper() # load data

nn = init_network([784, 30, 10])

for l in range(0, nn.num_layers - 1):
    print('\nNetwork layer {0}'.format(l + 2)) # disregard the input layer
    print(f"weights shape: {nn.weights[l].shape}")
    print(f"biases shape: {nn.biases[l].shape}")

# hyper parameters
epochs = 15
mini_batch_size = 10
learning_rate = 3.0

print('\nLearning process started...\n')

time_start = time()

learn(nn, training_data, epochs, mini_batch_size, learning_rate, test_data)

time_end = time()

time_elapsed = time_end - time_start

print('\nLearning process complete in {0} seconds ({1} seconds per epoch)!\n'.format(f'{time_elapsed:.0f}', f'{time_elapsed / epochs:.1f}'))

print('Validation (with yet unseen data): accuracy {0}%'.format(100.0 * evaluate(nn, validation_data) / len(validation_data)))


Network layer 2
weights shape: (30, 784)
biases shape: (30, 1)

Network layer 3
weights shape: (10, 30)
biases shape: (10, 1)

Learning process started...

Epoch  1: accuracy 89.93%
Epoch  2: accuracy 91.88%
Epoch  3: accuracy 92.53%
Epoch  4: accuracy 93.01%
Epoch  5: accuracy 93.33%
Epoch  6: accuracy 93.01%
Epoch  7: accuracy 93.22%
Epoch  8: accuracy 93.03%
Epoch  9: accuracy 93.63%
Epoch 10: accuracy 93.65%
Epoch 11: accuracy 93.87%
Epoch 12: accuracy 93.87%
Epoch 13: accuracy 93.56%
Epoch 14: accuracy 93.92%
Epoch 15: accuracy 94.0%

Learning process complete in 27 seconds (1.8 seconds per epoch)!

Validation (with yet unseen data): accuracy 94.85%


In [32]:
def load_image(file_name):
  digit=Image.open(file_name)
  digit=ImageOps.invert(digit)
  pixels=digit.load()
  return np.array(digit).reshape((784, 1))/255

In [33]:
def recognize_image(path, file):
    x = load_image(path.format(file))

    y = feedforward(nn, x)

    bitmap = x.reshape((28, 28))

    file_num = int(file)
    result = y.argmax()

    if file_num == result:
        ev = 'correctly'
    else:
        ev = 'incorrectly'

    print(file, 'was', ev, 'recognized as', result)

In [25]:
import zipfile
import os

zip_path = 'non-MNIST-digits.zip'
extract_path = './'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Files extracted successfully!")
else:
    print(f"Error: {zip_path} not found in your main directory. Make sure it's uploaded!")

Files extracted successfully!


In [26]:
print('Non-MNIST digits:\n')

for file in range(0,10):
    recognize_image('./non-MNIST-digits/{0}.png', file)

Non-MNIST digits:

0 was incorrectly recognized as 2
1 was correctly recognized as 1
2 was correctly recognized as 2
3 was correctly recognized as 3
4 was correctly recognized as 4
5 was correctly recognized as 5
6 was incorrectly recognized as 2
7 was incorrectly recognized as 1
8 was incorrectly recognized as 5
9 was correctly recognized as 9
